In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("load_type", "", "Load Type")
dbutils.widgets.text("merge_key", "", "Merge Key")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
load_type = dbutils.widgets.get("load_type")
merge_key = dbutils.widgets.get("merge_key")

In [0]:
bronze_df = spark.read.table(f"adb_newcaresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
from pyspark.sql import functions as F
updated_bronze_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("updated_timestamp", F.current_timestamp())

In [0]:
from delta.tables import DeltaTable

# Name of the silver (target) table
target_table = f"adb_newcaresync.silver.{table_name}"
merge_keys = [key.strip() for key in merge_key.split(",")]

if load_type == "FULL":
    updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type == "APPEND":
    updated_bronze_df.write.mode("append").saveAsTable(target_table)

elif load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        silver_table = DeltaTable.forName(spark, target_table)
        join_condition = " AND ".join([f"target.{k} = source.{k}" for k in merge_keys])
        (
            silver_table.alias("target")
            .merge(updated_bronze_df.alias("source"), join_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

else:
    raise ValueError(f"Invalid load_type: {load_type}. Use FULL, APPEND or MERGE.")

In [0]:
record_count = updated_bronze_df.count()
dbutils.notebook.exit(str(record_count))